In [1]:
from utils import *

In [1]:
# !conda info
!pip -V

pip 23.0.1 from /opt/miniconda3/envs/test/lib/python3.9/site-packages/pip (python 3.9)


In [9]:
import openai
import os
openai.api_key = os.environ.get("OPENAI_TOKEN")

In [4]:
from IPython.display import display_markdown
def display(content):
    display_markdown(f'''```sql
{content}
```''', raw=True)

In [5]:
def gen_sql(prompt):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0301",
        messages=[{"role": "assistant", "content": prompt}],
        max_tokens=800,
    )
    display(response.choices[0].message.content)
    return response

In [6]:
opt_tip = """这里有一些优化技巧，需要你在编写 sql 的过程中需要考虑，具体技巧如下：
1. 为了提高执行效率，你需要最小化数据库的操作数；
2. 尽量使用索引开加速连表（join）和过滤（where）操作；
3. 尽量在连表之前先进行聚合；
4. 如果存在分区字段，请保证查询的时候使用它；
5. 如果连表之前可以提前过滤和聚合，请尽量使用先使用子查询来降低连表数据；
6. 字段名和表名如果出现保留字，使用反引号(`)将字段名括起来。"""

In [7]:
opt_tip = """这里有一些优化 SQL 技巧你可以考虑：
1. SQL需要遵循 SQL-99 规范;
2. 字段名和表名如果出现保留字，使用反引号(`)将字段名括起来;
3. 为了优化查询速度，你需要最小化数据库的操作数。"""

In [10]:
filter_prompt = build_prompt("获取客户 A 的电话号码和住址", ["Customer"])
print(filter_prompt)
filter = gen_sql(filter_prompt)

现在你是一个资深的分析师，有着丰富分析经验和SQL能力，请你帮个忙

Customer 的表结构如下：
  id INTEGER COMMENT "主键"
  name VARCHAR(50)
  phone VARCHAR(15)
  address VARCHAR(100)    
Customer 的样本数据如下：
 id name       phone address
  1   小糕 13616161616      福州
  2    多      112323      福州
  3   张三 13888888888 上海市浦东新区
  4   李四 13999999999  北京市海淀区
  5   王五 13666666666  广州市天河区

根据以上提供的数据结构和数据，编写一个具体和准确的 mysql 的查询语句来回答如下分析问题：

"获取客户 A 的电话号码和住址"

并使用你的逻辑来注释它。 


```sql
SELECT phone, address  -- 选择要查询的列
FROM Customer
WHERE name = 'A'  -- 条件为客户名为 A
```

In [9]:
agg_prompt = build_prompt("统计每个月的订单数", ["Order"])
print(agg_prompt)
agg = gen_sql(agg_prompt)

现在你是一个资深的分析师，有着丰富分析经验和SQL能力，请你帮个忙

Order 的表结构如下：
  id INTEGER
  order_no VARCHAR(50)
  customer_id INTEGER
  payment_id INTEGER
  shipping_address_id INTEGER
  order_status_id INTEGER
  invoice_id INTEGER
  created_at DATETIME
  updated_at DATETIME    
Order 的样本数据如下：
 id    order_no  customer_id  payment_id  shipping_address_id  order_status_id  invoice_id          created_at          updated_at
  1 20220601001            1           1                    1                2         2.0 2022-06-01 09:00:00 2022-06-02 10:00:00
  2 20220601002            2           3                    2                1         NaN 2022-06-01 14:00:00 2022-06-02 16:00:00
  3 20220602001            3           2                    3                3         1.0 2022-06-02 08:00:00                 NaT

根据以上提供的数据结构和数据，编写一个具体和准确的 mysql 的查询语句来回答如下分析问题：

"统计每个月的订单数"

并使用你的逻辑来注释它。 


```sql


SELECT 
    DATE_FORMAT(created_at, '%Y-%m') AS month, 
    COUNT(*) AS order_count
FROM 
    Order
GROUP BY 
    month;

-- 将创建时间按年-月格式化，统计每个月的订单数，并按月份分组。
```

In [9]:
join_prompt = build_prompt("获取本月的所有订单，需要显示客户名和单号", ["Customer", "Order"])
print(join_prompt)
agg = gen_sql(join_prompt)

现在你是一个资深的分析师，有着丰富分析经验和SQL能力，请你帮个忙

Customer 的表结构如下：
  id INTEGER COMMENT "主键"
  name VARCHAR(50)
  phone VARCHAR(15)
  address VARCHAR(100)    
Customer 的样本数据如下：
 id name       phone address
  1   小糕 13616161616      福州
  2    多      112323      福州
  3   张三 13888888888 上海市浦东新区
  4   李四 13999999999  北京市海淀区
  5   王五 13666666666  广州市天河区

Order 的表结构如下：
  id INTEGER
  order_no VARCHAR(50)
  customer_id INTEGER
  payment_id INTEGER
  shipping_address_id INTEGER
  order_status_id INTEGER
  invoice_id INTEGER
  created_at DATETIME
  updated_at DATETIME    
Order 的样本数据如下：
 id    order_no  customer_id  payment_id  shipping_address_id  order_status_id  invoice_id          created_at          updated_at
  1 20220601001            1           1                    1                2         2.0 2022-06-01 09:00:00 2022-06-02 10:00:00
  2 20220601002            2           3                    2                1         NaN 2022-06-01 14:00:00 2022-06-02 16:00:00
  3 20220602001            3         

```sql


SELECT c.name, o.order_no
FROM Customer c
INNER JOIN Order o
ON c.id = o.customer_id
WHERE MONTH(o.created_at) = MONTH(CURRENT_DATE()) 
AND YEAR(o.created_at) = YEAR(CURRENT_DATE()); 

-- 使用内连接将 Customer 表和 Order 表连接起来
-- 在 WHERE 子句中筛选出本月的订单
-- 最后只选出客户名和单号这两个需要显示的字段
```

In [10]:
join_agg_prompt = build_prompt("统计每个月的手机号数量", ["Customer", "Order"])
print(join_agg_prompt)
join_agg = gen_sql(join_agg_prompt)

现在你是一个资深的分析师，有着丰富分析经验和SQL能力，请你帮个忙

Customer 的表结构如下：
  id INTEGER COMMENT "主键"
  name VARCHAR(50)
  phone VARCHAR(15)
  address VARCHAR(100)    
Customer 的样本数据如下：
 id name       phone address
  1   小糕 13616161616      福州
  2    多      112323      福州
  3   张三 13888888888 上海市浦东新区
  4   李四 13999999999  北京市海淀区
  5   王五 13666666666  广州市天河区

Order 的表结构如下：
  id INTEGER
  order_no VARCHAR(50)
  customer_id INTEGER
  payment_id INTEGER
  shipping_address_id INTEGER
  order_status_id INTEGER
  invoice_id INTEGER
  created_at DATETIME
  updated_at DATETIME    
Order 的样本数据如下：
 id    order_no  customer_id  payment_id  shipping_address_id  order_status_id  invoice_id          created_at          updated_at
  1 20220601001            1           1                    1                2         2.0 2022-06-01 09:00:00 2022-06-02 10:00:00
  2 20220601002            2           3                    2                1         NaN 2022-06-01 14:00:00 2022-06-02 16:00:00
  3 20220602001            3         

```sql


SELECT 
    DATE_FORMAT(created_at, '%Y-%m') AS `month`, -- 提取订单创建时间的月份
    COUNT(DISTINCT phone) AS phone_count -- 统计每月不同手机号的数量
FROM 
    Order o 
    JOIN Customer c ON o.customer_id = c.id -- 关联订单和顾客表
WHERE 
    c.phone IS NOT NULL -- 排除手机号为空的顾客
GROUP BY 
    `month` -- 按月份分组
ORDER BY 
    `month` ASC; -- 按月份升序排列

逻辑解析：
1. 通过JOIN操作关联订单表和顾客表，以获取订单所属顾客的手机号码信息。
2. 使用DATE_FORMAT()函数提取订单创建时间的月份信息。
3. 使用COUNT()函数和DISTINCT关键字统计每个月份下不同手机号的数量。
4. 使用WHERE子句过滤出手机号不为空的顾客。
5. 使用GROUP BY子句按月份对数据进行分组。
6. 使用ORDER BY子句按月份升序排列结果。
```

In [11]:
debug_prompt = f"""{join_agg.choices[0].message.content}

这个 SQL 产生 如下错误：

You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'Order o ON c.id = o.customer_id 
GROUP BY month 
ORDER BY month ASC' at line 3

找出问题，并重写一个新的 SQL
"""
print(debug_prompt)
join_agg = gen_sql(debug_prompt)



SELECT 
    DATE_FORMAT(created_at, '%Y-%m') AS `month`, -- 提取订单创建时间的月份
    COUNT(DISTINCT phone) AS phone_count -- 统计每月不同手机号的数量
FROM 
    Order o 
    JOIN Customer c ON o.customer_id = c.id -- 关联订单和顾客表
WHERE 
    c.phone IS NOT NULL -- 排除手机号为空的顾客
GROUP BY 
    `month` -- 按月份分组
ORDER BY 
    `month` ASC; -- 按月份升序排列

逻辑解析：
1. 通过JOIN操作关联订单表和顾客表，以获取订单所属顾客的手机号码信息。
2. 使用DATE_FORMAT()函数提取订单创建时间的月份信息。
3. 使用COUNT()函数和DISTINCT关键字统计每个月份下不同手机号的数量。
4. 使用WHERE子句过滤出手机号不为空的顾客。
5. 使用GROUP BY子句按月份对数据进行分组。
6. 使用ORDER BY子句按月份升序排列结果。

这个 SQL 产生 如下错误：

You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'Order o ON c.id = o.customer_id 
GROUP BY month 
ORDER BY month ASC' at line 3

找出问题，并重写一个新的 SQL



```sql


问题在于使用了不合法的表名Order，因为Order是MySQL的保留关键字，应该用反引号 ` 来包裹才能使用。因此，重写后的SQL语句如下：

SELECT 
    DATE_FORMAT(created_at, '%Y-%m') AS `month`, -- 提取订单创建时间的月份
    COUNT(DISTINCT phone) AS phone_count -- 统计每月不同手机号的数量
FROM 
    `Order` o 
    JOIN Customer c ON o.customer_id = c.id -- 关联订单和顾客表
WHERE 
    c.phone IS NOT NULL -- 排除手机号为空的顾客
GROUP BY 
    `month` -- 按月份分组
ORDER BY 
    `month` ASC; -- 按月份升序排列
```